# 2.9 — Joins on RDDs

**Chapter 2, section 2.9** (*Joins*).

**The question this notebook answers:** what does each of the four join flavours return, on
data small enough to check by eye?

A join combines two keyed data sets by matching records that share a key, producing one output
record **for every matching pair** — which is where the row counts come from, and where the
first surprise usually is. The four variants below are the ones a reader familiar with SQL will
recognize.

This is the reference for the operators. Notebook 2.8 is about avoiding the shuffle they cost.

Runs on a laptop in well under a minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
# Chapter 2, section 2.9.
import os, tempfile
from pyspark.sql import SparkSession

SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-2.9")
         .master("local[*]")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

sc = spark.sparkContext
print("Spark", spark.version, "on", sc.master)

Spark 4.2.0 on local[*]


## 1. The chapter's two RDDs

Note that `x` has the key `A` twice and `y` has it twice as well. That is deliberate.

In [2]:
x = sc.parallelize([('C', 4), ('B', 3), ('A', 2), ('A', 1)])
y = sc.parallelize([('A', 8), ('B', 7), ('A', 6), ('D', 5)])

print("x =", x.collect())
print("y =", y.collect())
print("\nkeys in x:", sorted(set(x.keys().collect())))
print("keys in y:", sorted(set(y.keys().collect())))

x = [('C', 4), ('B', 3), ('A', 2), ('A', 1)]
y = [('A', 8), ('B', 7), ('A', 6), ('D', 5)]



keys in x: ['A', 'B', 'C']
keys in y: ['A', 'B', 'D']


## 2. `join` — the keys present in both

An inner join. `C` is dropped because `y` has no `C`; `D` is dropped because `x` has no `D`.

`A` appears twice on each side, so the join emits **one row per matching pair**: 2 × 2 = 4
rows for `A` alone. This is the behaviour worth internalizing, because it is how a join of two
large tables can produce an output far larger than either input — which is the fourth of
chapter 1's cluster-justifying conditions, "the shuffle is genuinely wide".

In [3]:
inner = sorted(x.join(y).collect())
for row in inner:
    print(row)

print(f"\n{len(inner)} rows: "
      f"{sum(1 for r in inner if r[0] == 'A')} for A (2 x 2 pairs) and "
      f"{sum(1 for r in inner if r[0] == 'B')} for B (1 x 1)")
print("Neither input had more than four records, and the output has five.")

('A', (1, 6))
('A', (1, 8))
('A', (2, 6))
('A', (2, 8))
('B', (3, 7))

5 rows: 4 for A (2 x 2 pairs) and 1 for B (1 x 1)
Neither input had more than four records, and the output has five.


## 3. `leftOuterJoin` — everything from the left

All of `x`, with `None` wherever `y` has no match. `C` survives, paired with `None`.

In [4]:
for row in sorted(x.leftOuterJoin(y).collect(), key=str):
    print(row)

('A', (1, 6))
('A', (1, 8))
('A', (2, 6))
('A', (2, 8))
('B', (3, 7))
('C', (4, None))


## 4. `rightOuterJoin` — everything from the right

All of `y`, with `None` wherever `x` has no match. `D` survives.

In [5]:
for row in sorted(x.rightOuterJoin(y).collect(), key=str):
    print(row)

('A', (1, 6))
('A', (1, 8))
('A', (2, 6))
('A', (2, 8))
('B', (3, 7))
('D', (None, 5))


## 5. `fullOuterJoin` — every key from either side

`C` and `D` both survive, each with a `None` on the side that lacks them.

In [6]:
for row in sorted(x.fullOuterJoin(y).collect(), key=str):
    print(row)

('A', (1, 6))
('A', (1, 8))
('A', (2, 6))
('A', (2, 8))
('B', (3, 7))
('C', (4, None))
('D', (None, 5))


In [7]:
# The four side by side, as row counts.
print(f"{'join flavour':18s}{'rows':>6s}   keys present")
for name, rdd in (("join", x.join(y)),
                  ("leftOuterJoin", x.leftOuterJoin(y)),
                  ("rightOuterJoin", x.rightOuterJoin(y)),
                  ("fullOuterJoin", x.fullOuterJoin(y))):
    rows = rdd.collect()
    print(f"{name:18s}{len(rows):>6}   {sorted({k for k, _ in rows})}")

join flavour        rows   keys present


join                   5   ['A', 'B']


leftOuterJoin          6   ['A', 'B', 'C']
rightOuterJoin         6   ['A', 'B', 'D']


fullOuterJoin          7   ['A', 'B', 'C', 'D']


## 6. Dropping the key afterwards

A join returns `(key, (left_value, right_value))`. If the key was only ever the thing being
matched on, and is not wanted in the result, `map` it away.

In [8]:
paired = x.join(y).map(lambda kv: (kv[1][0], kv[1][1]))
print(sorted(paired.collect()))

# Or keep the key and flatten the nesting, which is usually more useful downstream.
flat = x.join(y).map(lambda kv: (kv[0], kv[1][0], kv[1][1]))
print(sorted(flat.collect()))

[(1, 6), (1, 8), (2, 6), (2, 8), (3, 7)]
[('A', 1, 6), ('A', 1, 8), ('A', 2, 6), ('A', 2, 8), ('B', 3, 7)]


## 7. Building a pair RDD with `zip`

Not a join, but it is how a keyed RDD is often assembled from two parallel sequences, and it
carries a restriction worth knowing: `zip` requires both RDDs to have the **same number of
partitions** *and* the **same number of elements in each partition**. It is therefore only
safe on RDDs derived from a common ancestor, or built identically — which is why it is rare in
practice and common in examples.

In [9]:
valueRDDA = sc.parallelize(["a", "b", "c", "d", "e"])
valueRDDB = sc.parallelize(["AA", "BB", "CC", "DD"])
keysB = sc.parallelize([1, 1, 5, 2, 3])
keysC = sc.parallelize([1, 5, 5, 6])

rdd1 = keysB.zip(valueRDDA)
rdd2 = keysC.zip(valueRDDB)
print("rdd1 :", rdd1.collect())
print("rdd2 :", rdd2.collect())

print("\njoin :", sorted(rdd1.join(rdd2).collect(), key=str))
print("\nKey 1 appears twice in rdd1 and once in rdd2 -> 2 rows.")
print("Key 5 appears once in rdd1 and twice in rdd2 -> 2 rows.")
print("Key 2 and 3 are only in rdd1, key 6 only in rdd2 -> dropped by the inner join.")

rdd1 : [(1, 'a'), (1, 'b'), (5, 'c'), (2, 'd'), (3, 'e')]
rdd2 : [(1, 'AA'), (5, 'BB'), (5, 'CC'), (6, 'DD')]



join : [(1, ('a', 'AA')), (1, ('b', 'AA')), (5, ('c', 'BB')), (5, ('c', 'CC'))]

Key 1 appears twice in rdd1 and once in rdd2 -> 2 rows.
Key 5 appears once in rdd1 and twice in rdd2 -> 2 rows.
Key 2 and 3 are only in rdd1, key 6 only in rdd2 -> dropped by the inner join.


In [10]:
# What zip refuses to do.  This cell fails ON PURPOSE: the restriction is the lesson.
#
# The failure happens on an EXECUTOR, so Spark logs the whole JVM stack trace at ERROR
# level before the exception ever reaches Python.  The log level is lowered around the
# call to keep that out of the output; this is the standard way to show a deliberate
# executor-side failure in a notebook that has to run headless.
sc.setLogLevel("FATAL")
try:
    bad = sc.parallelize([1, 2, 3], 2).zip(sc.parallelize(["a", "b", "c", "d"], 2))
    print(bad.collect())
except Exception as e:
    cause = next((ln.strip().lstrip(": ") for ln in str(e).splitlines()
                  if "SparkException" in ln and "zip" in ln), str(e).splitlines()[0])
    print(f"{type(e).__name__}")
    print(f"  {cause}")
finally:
    sc.setLogLevel("ERROR")

print("\nzip cannot pair RDDs whose partitions do not line up, and it finds out only")
print("when the data is read -- which is a good reminder that laziness defers errors")
print("as well as work.  The two RDDs above have the same partition count and differing")
print("element counts, and nothing complained until collect() asked for a result.")

Py4JJavaError
  org.apache.spark.SparkException: Can only zip RDDs with same number of elements in each partition

zip cannot pair RDDs whose partitions do not line up, and it finds out only
when the data is read -- which is a good reminder that laziness defers errors
as well as work.  The two RDDs above have the same partition count and differing
element counts, and nothing complained until collect() asked for a result.


## Conclusion

| flavour | keeps | `None` appears |
|---|---|---|
| `join` | keys in **both** | never |
| `leftOuterJoin` | all of the **left** | on the right, for unmatched left keys |
| `rightOuterJoin` | all of the **right** | on the left, for unmatched right keys |
| `fullOuterJoin` | keys from **either** | on whichever side lacks the key |

Two things to carry forward.

**A join emits one row per matching pair.** Duplicate keys multiply: two `A`s on each side give
four rows. On real data this is how an output grows larger than either input, and it is the
reason a join is the operation most likely to exhaust a cluster.

**A join is a wide dependency.** Both sides are shuffled so that matching keys meet, and on
large inputs it is usually the most expensive operation in the job. Which is exactly why
notebook 2.8 exists: when one side is small, the shuffle can be removed entirely.

**Next.** Chapter 3 moves to the DataFrame API, where the same joins are written declaratively
and the optimizer chooses the physical strategy — including the broadcast join of 2.8 — on your
behalf.